# This hands on practice to do fine-tune any LLM.

2 types of fine-tune:
1. Full fine tune
2. LoRA or QLoRA (quantized) based fine tuning

QLoRA, why need quantized model?
- For example, if I have Llama 7B param, now if takes 4bytes (high precision like float32) then it will need 28GB of our memory, that means harder to run on local machine due to less number of GPU.
- So, if we quantized this 4bytes to 1bytes (low precision like int8) then it will take 7GB, much smaller.\

Finetune with Unsloth

In [2]:
!pip install -q unsloth
!pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.

Model used: Llama-3.2

In [3]:
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
max_seq_length = 2048 # Choose any to increase or decrease context window! Unsloth also supports RoPE (Rotary Positinal Embedding) scaling internally.
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

`model_name`:Specifies the name of the pre-trained model to load.

`max_seq_length`:Defines the maximum sequence length (in tokens) that the model can process. max_seq_length = 2048 allows the model to process sequences up to 2048 tokens long.

`dtype`:Specifies the data type for model weights and computations. None: Automatically selects the appropriate data type based on the hardware. torch.

`float16`: Uses 16-bit floating point precision, reducing memory usage and potentially increasing speed on compatible GPUs. torch.bfloat16: Similar to float16 but with a wider dynamic range, beneficial for certain hardware like NVIDIA A100 GPUs.

`load_in_4bi`t:Determines whether to load the model using 4-bit quantization.Ideal for scenarios where memory efficiency is crucial, such as deploying models on edge devices or during experimentation.

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit, # Will load the 4Bit Quantized Model
)

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


# Define QLoRA

Now, we'll use the `get_peft_model` from unsloth's `FastLanguageModel` class to attach adapters (peft layers) on top of the models in order to perform QLoRA


`r`: The rank of the low-rank matrices in LoRA; higher values can capture more information but increase memory usage.

`target_modules`: List of model components (e.g., "q_proj", "k_proj") where LoRA adapters are inserted for fine-tuning.

`lora_alpha`: Scaling factor for the LoRA updates; controls the impact of the adapters on the model's outputs.

`lora_dropout`: Dropout rate applied to LoRA layers during training to prevent overfitting. (regularization)

`bias`: Specifies how biases are handled in LoRA layers; options include "none", "all", or "lora_only".

`use_gradient_checkpointing`: Enables gradient checkpointing to reduce memory usage during training; "unsloth" uses Unsloth's optimized version.

`random_state`: Seed for random number generators to ensure reproducibility of training results.

`use_rslora`: Boolean indicating whether to use Rank-Stabilized LoRA (rsLoRA) for potentially more stable training.

`loftq_config`: Configuration for Low-Rank Quantization (LoftQ); set to None to disable this feature.

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16, # a higher alpha value assigns more weight to the LoRA activations
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Train HF dataset

In [7]:
from datasets import load_dataset
dataset = load_dataset("ServiceNow-AI/R1-Distill-SFT",'v0', split = "train")

README.md: 0.00B [00:00, ?B/s]

v0/train-00000-of-00003.parquet:   0%|          | 0.00/180M [00:00<?, ?B/s]

v0/train-00001-of-00003.parquet:   0%|          | 0.00/187M [00:00<?, ?B/s]

v0/train-00002-of-00003.parquet:   0%|          | 0.00/188M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/171647 [00:00<?, ? examples/s]

In [8]:
print(dataset[:5])

{'id': ['id_0', 'id_1', 'id_2', 'id_3', 'id_4'], 'reannotated_assistant_content': ['<think>\nFirst, I need to determine the total number of children on the playground by adding the number of boys and girls.\n\nThere are 27 boys and 35 girls.\n\nAdding these together: 27 boys + 35 girls = 62 children.\n\nTherefore, the total number of children on the playground is 62.\n</think>\n\nTo find the total number of children on the playground, we simply add the number of boys and girls together.\n\n\\[\n\\text{Total children} = \\text{Number of boys} + \\text{Number of girls}\n\\]\n\nPlugging in the given values:\n\n\\[\n\\text{Total children} = 27 \\text{ boys} + 35 \\text{ girls} = 62 \\text{ children}\n\\]\n\n**Final Answer:**\n\n\\[\n\\boxed{62}\n\\]', '<think>\nFirst, I need to determine the cost per dozen oranges. John bought three dozen oranges for \\$28.80, so I can find the cost per dozen by dividing the total cost by the number of dozens.\n\nNext, with the cost per dozen known, I can 

Now, we create a prompt template that will be used to finetune our Llama model

In [9]:
r1_prompt = """You are a thinking asssistant to make understanding of user question, think like a human. You should not make self-judgement, you will make based on what you learned to coming up well established responses as an answer.
<problem>
{}
</problem>

{}
{}
"""
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
  problems = examples["problem"]
  thoughts = examples["reannotated_assistant_content"]
  solutions = examples["solution"]
  texts = []

  for problem, thought, solution in zip(problems, thoughts, solutions):
    text = r1_prompt.format(problem, thought, solution)+EOS_TOKEN
    texts.append(text)

  return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/171647 [00:00<?, ? examples/s]

# Trainer Setup:

`model and tokenizer`: These are the model and tokenizer objects that will be trained.

`train_dataset`: The dataset used for training.

`dataset_text_field`: Specifies the field in the dataset that contains the text data.

`max_seq_length`: Maximum sequence length for the input data.

`dataset_num_proc`: Number of processes to use for data loading.

`packing`: If True, enables sequence packing (concatenates multiple examples into a single sequence to better utilize tokens).

# Training Arguments:

`per_device_train_batch_size`: Number of samples per batch for each device.

`gradient_accumulation_steps`: Number of steps to accumulate gradients before updating model weights.

`warmup_steps`: Number of steps for learning rate warmup.

`max_steps`: Total number of training steps.

`learning_rate`: Learning rate for the optimizer.

`fp16 and bf16`: Specifies whether to use 16-bit floating point precision or bfloat16, depending on hardware support.

`logging_steps`: Frequency of logging training progress.

`optim`: Optimizer type, here using an 8-bit version of AdamW.

`weight_decay`: Regularization parameter for weight decay.

`lr_scheduler_type`: Type of learning rate scheduler.

`seed`: Random seed for reproducibility.

`output_dir`: Directory where the training outputs will be saved.

`report_to`: Integration for observability tools like "wandb", "tensorboard", etc.

In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model, # base lora model
    tokenizer = tokenizer, # trainer will load dataset to train the model
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2, # Number of processors to use for processing the dataset
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2, # The batch size per GPU/TPU core
        gradient_accumulation_steps = 4, # Number of steps to perform befor each gradient accumulation
        warmup_steps = 5, # Few updates with low learning rate before actual training
        max_steps = 60, # Specifies the total number of training steps (batches) to run.
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # Optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc for observability
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/171647 [00:00<?, ? examples/s]

In [11]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 171,647 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.078514
2,1.006949
3,1.110829
4,1.021830
5,0.863045
6,0.937021
7,0.834719
8,0.816367
9,0.854687
10,0.804540


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [12]:
from unsloth.chat_templates import get_chat_template
sys_prompt = """You are a thinking asssistant to make understanding of user question, think like a human. You should not make self-judgement, you will make based on what you learned to coming up well established responses as an answer.
<problem>
{}
</problem>
"""
message = sys_prompt.format("How many 'r's are present in 'strawberry'?")
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": message},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 1024, use_cache = True,
                         temperature = 1.5, min_p = 0.1)
response = tokenizer.batch_decode(outputs)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_att

In [13]:
print(response[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

You are a thinking asssistant to make understanding of user question, think like a human. You should not make self-judgement, you will make based on what you learned to coming up well established responses as an answer.
<problem>
How many 'r's are present in'strawberry'?
</problem>
<|eot_id|><|start_header_id|>assistant<|end_header_id|>

To find out how many 'r's are present in'strawberry', I would count each letter individually.

1. Start from the first letter:'s'. No 'r'.

2. Move on to the next letter: 't'. No 'r'.

3. The third letter is 'r'. Yes, it's an 'r'.

4. Fourth letter: 'a'. No 'r'.

5. Fifth letter: 'w'. No 'r'.

6. Sixth letter: 'b'. No 'r'.

7. Seventh letter: 'e'. No 'r'.

8. Eighth letter: 'r'. Yes, it's another 'r'.

So, there are two 'r's in'strawberry'.<|eot_id|>


In [14]:
model.save_pretrained("CustomLLM-3B")  # Local saving
tokenizer.save_pretrained("CustomLLM-3B")

Unsloth: Restored added_tokens_decoder metadata in CustomLLM-3B/tokenizer_config.json.


('CustomLLM-3B/tokenizer_config.json',
 'CustomLLM-3B/chat_template.jinja',
 'CustomLLM-3B/tokenizer.json')

In [15]:
model.save_pretrained_gguf("CustomLLM-3B-GGUF", tokenizer,)

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in CustomLLM-3B-GGUF/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:28<00:00, 14.10s/it]


Unsloth: Merge process complete. Saved to `/content/CustomLLM-3B-GGUF`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['CustomLLM-3B-GGUF_gguf/llama-3.2-3b-instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q8_0. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['CustomLLM-3B-GGUF_gguf/llama-3.2-3b-instruct.Q8_0.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model CustomLLM-3B-GGUF_gguf/llama-3.2-3b-instruct.Q8_0.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to CustomLLM-3B-GGUF_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f CustomLLM-3B-GGUF_gguf/Modelfile


{'save_directory': 'CustomLLM-3B-GGUF',
 'gguf_directory': 'CustomLLM-3B-GGUF_gguf',
 'gguf_files': ['CustomLLM-3B-GGUF_gguf/llama-3.2-3b-instruct.Q8_0.gguf'],
 'modelfile_location': 'CustomLLM-3B-GGUF_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

Running the model via Ollama (OPTIONAL)

In [20]:
!sudo apt-get update
!sudo apt-get install -y zstd pciutils

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [21]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [25]:
import subprocess
import time
import threading

# Start Ollama server in background
def run_ollama_server():
    subprocess.run(['ollama', 'serve'])

# Start server in a separate thread
server_thread = threading.Thread(target=run_ollama_server, daemon=True)
server_thread.start()

# Wait for server to initialize
time.sleep(5)
print("✅ Ollama server started!")

✅ Ollama server started!


In [26]:
print(tokenizer._ollama_modelfile)


FROM {__FILE_LOCATION__}
TEMPLATE """{{ if .Messages }}
{{- if or .System .Tools }}<|start_header_id|>system<|end_header_id|>
{{- if .System }}

{{ .System }}
{{- end }}
{{- if .Tools }}

You are a helpful assistant with tool calling capabilities. When you receive a tool call response, use the output to format an answer to the original use question.
{{- end }}
{{- end }}<|eot_id|>
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 }}
{{- if eq .Role "user" }}<|start_header_id|>user<|end_header_id|>
{{- if and $.Tools $last }}

Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.

Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}. Do not use variables.

{{ $.Tools }}
{{- end }}

{{ .Content }}<|eot_id|>{{ if $last }}<|start_header_id|>assistant<|end_header_id|>

{{ end }}
{{- else if eq .Role "assistant" }}<|start_header

In [30]:
!find . -name "Modelfile" -o -name "*.gguf"

./CustomLLM-3B-GGUF_gguf/Modelfile
./CustomLLM-3B-GGUF_gguf/llama-3.2-3b-instruct.Q8_0.gguf


In [31]:
!ollama create unsloth_model -f ./CustomLLM-3B-GGUF_gguf/Modelfile

In [32]:
!ollama list

NAME                    ID              SIZE      MODIFIED       
unsloth_model:latest    1bfadf72a688    3.4 GB    20 seconds ago    


In [35]:
!ollama run unsloth_model:latest "Hello" && nvidia-smi

How are you doing today? Is there something I can help you with, or would y
you like to chat?

Tue May 12 18:40:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             93W /  400W |   21775MiB /  81920MiB |     42%      Default |
|                                         |  

In [37]:
import requests
import json

# Ollama API endpoint
url = "http://localhost:11434/api/generate"

payload = {
    "model": "unsloth_model:latest",
    "prompt": "Explain a puzzle game to child",
    "stream": False
}

response = requests.post(url, json=payload)
result = response.json()
print(result['response'])

Oh boy, do I have something fun to explain!

Imagine you're playing with a brain teaser called "Logic Blocks". Here's how it works:

**The Game:**
You need to build a path for a little character named Max. Max is trying to reach the other side of the board from his starting point (marked A). He can move in four directions: up, down, left, and right.

But, oh no! Some blocks on the board are special because they can only be crossed in certain ways. You need to figure out how to build a path for Max using these rules:

* Each block can only be crossed by stepping on it once.
* You cannot cross any block more than once.
* You must reach the destination point (marked B).

**The Challenge:**
Your task is to find the shortest and simplest path from A to B. It's like solving a puzzle!

Here are some examples of blocks that need special attention:

* **Red blocks**: These blocks can only be crossed by stepping on them in an odd number of steps.
* **Blue blocks**: These blocks can only be cross

In [38]:
!ollama show unsloth_model

  Model
    architecture        llama     
    parameters          3.2B      
    context length      131072    
    embedding length    3072      
    quantization        Q8_0      

  Capabilities
    completion    
    tools         

  Parameters
    temperature    1.5                      
    min_p          0.1                      
    stop           "<|start_header_id|>"    
    stop           "<|end_header_id|>"      
    stop           "<|eot_id|>"             
    stop           "<|eom_id|>"             



In [39]:
!ollama run unsloth_model "Explain GPU acceleration" --verbose

GPU (Graphics Processing Unit) acceleration is a technology that allows sof
software applications to utilize the processing power of graphics processin
processing units (GPUs) to accelerate various tasks beyond just rendering 2
2D and 3D graphics. GPUs are designed to handle parallel computations, maki
making them particularly well-suited for certain types of workloads.

**History**

The concept of GPU acceleration emerged in the early 2000s, as the performa
performance gap between CPUs (Central Processing Units) and GPUs became app
apparent. Initially, GPUs were optimized for 3D graphics rendering, which r
required massive parallel processing capabilities to handle complex geometr
geometry calculations and lighting simulations. However, as computing requi
requirements evolved, developers recognized that other tasks could also ben
benefit from GPU acceleration.

**How GPU Acceleration Works**

GPU acceleration involves transferring a portion of a workload to the GPU, 
where it can be e